In [ ]:
import os
path = os.path.expanduser("~/sapiens_host/pose/torchscript/sapiens_0.3b")
# path = os.path.expanduser("~/sapiens-hvm/models/sapiens_seg_0.3b")
print(os.listdir(path))

['README.md', '.cache', '.gitattributes', 'sapiens_0.3b_goliath_best_goliath_AP_573_torchscript.pt2']


In [2]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [4]:
model_path = str(Path.home() / "sapiens_host/pose/torchscript/sapiens_0.3b/sapiens_0.3b_goliath_best_goliath_AP_573_torchscript.pt2")
model = torch.jit.load(model_path, map_location="cpu")
model.eval()
print("Model loaded")

Model loaded


In [7]:
VIDEO = Path("video/030820241100/p14_front_1.mp4")
cap = cv2.VideoCapture(str(VIDEO))
ok, frame = cap.read()
cap.release()

if not ok:
    raise RuntimeError(f"Could not read {VIDEO}")

# BGR -> RGB
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

# Resize to model expected input (W, H)
frame_resized = cv2.resize(frame_rgb, (768, 1024))

# Normalize with ImageNet mean/std
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])
frame_norm = (frame_resized / 255.0 - mean) / std

# HWC -> NCHW tensor
x = torch.tensor(frame_norm, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0)
print("Input shape:", x.shape)

Input shape: torch.Size([1, 3, 1024, 768])


In [8]:
with torch.no_grad():
    output = model(x)

print("Output type:", type(output))
print("Output shape:", output.shape if hasattr(output, 'shape') else [o.shape for o in output])

Output type: <class 'torch.Tensor'>
Output shape: torch.Size([1, 308, 256, 192])


In [9]:
heatmaps = output[0]  # (308, 256, 192)

# Get the peak location of each heatmap
keypoints = []
for i in range(308):
    heatmap = heatmaps[i]
    idx = torch.argmax(heatmap)
    y, x = divmod(idx.item(), heatmap.shape[1])
    
    # Scale back to original frame size
    x_orig = int(x * frame.shape[1] / 192)
    y_orig = int(y * frame.shape[0] / 256)
    confidence = heatmap[y, x].item()
    
    keypoints.append((x_orig, y_orig, confidence))

print(f"Decoded {len(keypoints)} keypoints")

Decoded 308 keypoints


In [ ]:
vis = frame.copy()

for x, y, conf in keypoints:
    if conf > 0.3:  # confidence threshold
        cv2.circle(vis, (x, y), 3, (0, 255, 0), -1)

plt.figure(figsize=(10, 12))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Sapiens Pose — 308 Keypoints")
plt.show()